DPI suffix & extention removal

In [6]:
import os
import re
import shutil
from tqdm import tqdm

# Paths
INPUT_DIR = r"D:\Y4 Research\datasets\ingredient & nutrition Images\png_converted"
OUTPUT_DIR = r"D:\Y4 Research\datasets\ingredient & nutrition Images\.400 removed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Regex: remove `.400` ONLY if it is right before the file extension
# Example matched: name.400.png → name.png
REMOVE_400_REGEX = re.compile(r"\.400(?=\.)")

# Get image files
image_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

for filename in tqdm(image_files, desc="Removing .400 from filenames", unit="image"):
    src_path = os.path.join(INPUT_DIR, filename)

    # Remove `.400` only if it exists before extension
    new_filename = REMOVE_400_REGEX.sub("", filename)
    dst_path = os.path.join(OUTPUT_DIR, new_filename)

    try:
        shutil.copy2(src_path, dst_path)
    except Exception as e:
        print(f"✖ Failed {filename}: {e}")


Removing .400 from filenames: 100%|██████████| 434/434 [00:03<00:00, 117.30image/s]


✅ Normalizes bounding boxes (0–1000)
✅ Filters low confidence OCR noise

In [ ]:
import os
import json
from PIL import Image
from tqdm import tqdm

# PATHS
OCR_INPUT_DIR = r"D:\Y4 Research\datasets\ocr_json\ing_nut_set1\new\DPI_suffix_removed"
OCR_OUTPUT_DIR = r"D:\Y4 Research\datasets\ocr_json\ing_nut_set1\new\cleaned_OCR"
IMAGE_DIR = r"D:\Y4 Research\datasets\ingredient & nutrition Images\.400 removed"

MIN_CONF = 40
os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------
# Build image index (base_name -> full path)
# ---------------------------------------------------
image_index = {}

for img in os.listdir(IMAGE_DIR):
    if img.lower().endswith((".png", ".jpg", ".jpeg")):
        base = os.path.splitext(img)[0]
        image_index[base] = os.path.join(IMAGE_DIR, img)

print(f"✔ Indexed {len(image_index)} images")


# ---------------------------------------------------
# Function to clean OCR & normalize boxes
# ---------------------------------------------------
def prepare_layoutlmv3_input(ocr_json, image_id, image_path, min_conf=40):
    image = Image.open(image_path).convert("RGB")
    W, H = image.size

    tokens, boxes = [], []

    for w in ocr_json.get("words", []):
        if w.get("confidence", 0) < min_conf:
            continue

        x, y, bw, bh = w["bbox"]

        box = [
            int((x / W) * 1000),
            int((y / H) * 1000),
            int(((x + bw) / W) * 1000),
            int(((y + bh) / H) * 1000),
        ]

        tokens.append(w["text"])
        boxes.append(box)

    # Return JSON-safe dict with image_id
    return {
        "image_id": image_id,
        "tokens": tokens,
        "bboxes": boxes
    }


# ---------------------------------------------------
# Batch processing
# ---------------------------------------------------
ocr_files = [f for f in os.listdir(OCR_INPUT_DIR) if f.endswith(".json")]
missing_images = []

for ocr_file in tqdm(ocr_files, desc="Cleaning OCR files"):
    ocr_path = os.path.join(OCR_INPUT_DIR, ocr_file)

    with open(ocr_path, "r", encoding="utf-8") as f:
        ocr_json = json.load(f)

    base_name = os.path.splitext(ocr_file)[0]
    image_path = image_index.get(base_name)

    if image_path is None:
        missing_images.append(base_name)
        continue

    # Pass image_id to the function
    cleaned_data = prepare_layoutlmv3_input(
        ocr_json=ocr_json,
        image_id=base_name,
        image_path=image_path,
        min_conf=MIN_CONF
    )

    output_path = os.path.join(OCR_OUTPUT_DIR, ocr_file)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------
# Report
# ---------------------------------------------------
print(f"\n✔ Completed")
print(f"✔ OCR processed: {len(ocr_files) - len(missing_images)}")
print(f"⚠ Missing images: {len(missing_images)}")


✔ Indexed 434 images


Cleaning OCR files:   0%|          | 0/434 [00:00<?, ?it/s]


TypeError: Object of type Image is not JSON serializable